# Hospital W 2.0 — Fase II: Procesamiento analítico distribuido con PySpark

Pipeline de Big Data sobre la telemetría de los sensores de pacientes (Figura 6 de la guía):

| Paso | Requisito de la guía |
|---|---|
| 1 | **Ecosistema nube** — inicializar un objeto `SparkSession` |
| 2 | **Ingesta masiva** — simular `logs_monitoreo_pacientes.csv` con ≥ 5,000 registros |
| 3 | **Query Big Data 1** — limpieza y filtrado de anomalías |
| 4 | **Query Big Data 2** — top 5 de alertas clínicas críticas con Spark SQL |
| 5 | **Query Big Data 3** — clasificación automática de riesgo con una UDF lambda |
| 6 | Nivel de riesgo sostenido por paciente (en el sistema se guarda en `asignacionCama.nivelRiesgo`) |
| 7 | **Exportación corporativa** — `.write.parquet()` |

Los logs **no se guardan en la base de datos**: viven solo en el CSV y en el Parquet refinado.

**Cómo correrlo en Google Colab:** *Archivo → Subir notebook* → elegir este archivo →
*Entorno de ejecución → Ejecutar todas*. No necesita archivos externos: los datos se generan en la celda 2.

## 1. Ecosistema nube: SparkSession

In [ ]:
# Colab ya trae PySpark en las versiones recientes; esta línea lo instala si faltara.
!pip install -q pyspark

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql import types as T

spark = (SparkSession.builder
         .appName("HospitalW-BigData")
         .config("spark.sql.shuffle.partitions", "8")
         .getOrCreate())

print("Spark", spark.version, "| master:", spark.sparkContext.master)
spark

## 2. Ingesta masiva: simulación de la telemetría

Se generan **6,000 lecturas** (mínimo pedido: 5,000) con el esquema de la Figura 6, de los
pacientes hospitalizados durante la última semana (en el dashboard salen de `asignacionCama`;
aquí, sin base de datos, se simulan 12 pacientes internados).
Cada paciente tiene un perfil (estable, inestable o crítico) y se inyectan a propósito
~2 % de valores nulos (sensor desconectado) y ~3 % de lecturas imposibles (falla del sensor),
que son las que debe remover la Query BD 1.

In [ ]:
import csv
import datetime as dt
import random
from pathlib import Path

RUTA_POR_DEFECTO = Path("logs_monitoreo_pacientes.csv")
COLUMNAS = ["ID_Registro", "ID_Paciente", "Timestamp", "Frecuencia_Cardiaca", "Saturacion_Oxigeno",
            "Alerta_Generada"]
DIAS_DE_MONITOREO = 7

# perfil -> (FC media, FC desvío, SpO2 media, SpO2 desvío)
PERFILES = {
    "estable": (76, 9, 96.5, 1.5),
    "inestable": (92, 16, 93.0, 2.8),
    "critico": (108, 22, 89.5, 3.5),
}


def generar(registros: int = 6000, estancias: list | None = None, ruta: Path = RUTA_POR_DEFECTO,
            semilla: int = 2026, fin: dt.datetime | None = None) -> dict:
    """
    estancias: lista de (ID_Paciente, desde, hasta) = período en que el paciente
    estuvo en cama. Si no se pasa (por ejemplo en Colab, sin base de datos), se
    simulan 12 pacientes internados durante toda la semana.
    """
    rnd = random.Random(semilla)
    fin = (fin or dt.datetime.now()).replace(second=0, microsecond=0)
    inicio = fin - dt.timedelta(days=DIAS_DE_MONITOREO)
    if not estancias:
        estancias = [(f"PAC-{i:04d}", inicio, None) for i in range(1, 13)]
    # Recorta cada estancia a la ventana de monitoreo (últimos 7 días).
    ventanas = []
    for paciente, desde, hasta in estancias:
        desde, hasta = max(desde, inicio), min(hasta or fin, fin)
        if hasta > desde:
            ventanas.append((paciente, desde, hasta))
    if not ventanas:
        raise ValueError("Ninguna internación cae dentro de los últimos 7 días: no hay a quién monitorear.")

    # Perfil por paciente: varios estables, algunos inestables y pocos críticos.
    perfil = {p: rnd.choices(list(PERFILES), weights=[40, 35, 25])[0] for p, _, _ in ventanas}
    # Las estancias más largas generan más lecturas (el sensor mide a intervalos regulares).
    pesos = [(hasta - desde).total_seconds() for _, desde, hasta in ventanas]

    lecturas = []
    for _ in range(registros):
        paciente, desde, hasta = rnd.choices(ventanas, weights=pesos)[0]
        momento = desde + (hasta - desde) * rnd.random()
        fc_media, fc_desvio, spo2_media, spo2_desvio = PERFILES[perfil[paciente]]
        lecturas.append([paciente, momento.replace(microsecond=0),
                         round(rnd.gauss(fc_media, fc_desvio)), min(100, round(rnd.gauss(spo2_media, spo2_desvio)))])
    lecturas.sort(key=lambda l: l[1])       # el archivo queda en orden cronológico

    ruta = Path(ruta)
    ruta.parent.mkdir(parents=True, exist_ok=True)
    nulos = fallas = alertas = 0
    with ruta.open("w", newline="", encoding="utf-8") as archivo:
        escritor = csv.writer(archivo)
        escritor.writerow(COLUMNAS)
        for i, (paciente, momento, fc, spo2) in enumerate(lecturas, start=1):
            sorteo = rnd.random()
            if sorteo < 0.03:                  # falla del sensor: lectura fisiológicamente imposible
                fallas += 1
                fc, spo2 = rnd.choice([(0, spo2), (rnd.randint(5, 29), spo2), (rnd.randint(221, 320), spo2),
                                       (fc, rnd.randint(101, 130))])
            elif sorteo < 0.05:                # sensor desconectado: campo vacío
                nulos += 1
                if rnd.random() < 0.5:
                    fc = None
                else:
                    spo2 = None
            alerta = "Sí" if (spo2 is not None and spo2 < 90) or (fc is not None and (fc > 120 or fc < 50)) else "No"
            alertas += alerta == "Sí"
            escritor.writerow([f"LOG-{i:06d}", paciente, momento.strftime("%Y-%m-%d %H:%M:%S"),
                               "" if fc is None else fc, "" if spo2 is None else spo2, alerta])
    return {"ruta": str(ruta), "registros": registros, "pacientes": len(perfil),
            "nulos_inyectados": nulos, "fallas_inyectadas": fallas, "alertas": alertas}

resumen = generar(registros=6000)
resumen

In [ ]:
# Lectura con esquema explícito (Figura 6): Spark no tiene que adivinar los tipos.
ESQUEMA = T.StructType([
    T.StructField("ID_Registro", T.StringType(), True),
    T.StructField("ID_Paciente", T.StringType(), True),
    T.StructField("Timestamp", T.TimestampType(), True),
    T.StructField("Frecuencia_Cardiaca", T.IntegerType(), True),
    T.StructField("Saturacion_Oxigeno", T.IntegerType(), True),
    T.StructField("Alerta_Generada", T.StringType(), True),
])

logs = (spark.read
        .option("header", True)
        .option("timestampFormat", "yyyy-MM-dd HH:mm:ss")
        .schema(ESQUEMA)
        .csv("logs_monitoreo_pacientes.csv"))

logs.printSchema()
print("Registros leídos:", logs.count())
logs.show(5, truncate=False)

## 3. Query Big Data 1 — Limpieza y filtrado de anomalías

Se remueven los registros con **valores nulos** y las **lecturas atípicas** que representan
fallas del sensor: frecuencia cardíaca **< 30 o > 220 lpm** y saturación fuera de 50–100 %.
Cada registro descartado recibe **un solo motivo** (el primero que cumple), así que la suma
de los motivos es exactamente el total de descartados.

In [ ]:
FC_MIN, FC_MAX = 30, 220
SPO2_MIN, SPO2_MAX = 50, 100

from pyspark.sql import Window

# Motivo de descarte de cada fila (NULL = la lectura es válida). Un solo motivo por fila.
repetido = F.row_number().over(Window.partitionBy("ID_Registro").orderBy("Timestamp")) > 1
motivo = (F.when(F.col("Frecuencia_Cardiaca").isNull() | F.col("Saturacion_Oxigeno").isNull()
                 | F.col("ID_Paciente").isNull() | F.col("Timestamp").isNull(), "Valores nulos (sensor desconectado)")
           .when(F.col("Frecuencia_Cardiaca") < FC_MIN, "FC < 30 lpm (falla del sensor)")
           .when(F.col("Frecuencia_Cardiaca") > FC_MAX, "FC > 220 lpm (falla del sensor)")
           .when(~F.col("Saturacion_Oxigeno").between(SPO2_MIN, SPO2_MAX), "SpO2 fuera de 50–100 % (falla del sensor)")
           .when(repetido, "Registro duplicado"))

marcado = logs.withColumn("motivo_descarte", motivo).cache()
print("Registros descartados por motivo:")
marcado.filter(F.col("motivo_descarte").isNotNull()).groupBy("motivo_descarte").count().show(truncate=False)

logs_limpios = marcado.filter(F.col("motivo_descarte").isNull()).drop("motivo_descarte")
descartados = marcado.filter(F.col("motivo_descarte").isNotNull()).count()
print(f"Leídos: {logs.count():,}  |  Descartados: {descartados:,}  |  Válidos: {logs_limpios.count():,}")

## 4. Query Big Data 2 — Top 5 de alertas clínicas críticas (Spark SQL)

Sobre una **vista temporal** de los logs limpios: los 5 pacientes únicos con más lecturas de
**saturación de oxígeno < 90 %** durante el período de monitoreo.

In [ ]:
logs_limpios.createOrReplaceTempView("logs_limpios")

top5 = spark.sql("""
    SELECT ID_Paciente,
           COUNT(*)                         AS alertas_criticas,
           MIN(Saturacion_Oxigeno)          AS spo2_minima,
           ROUND(AVG(Saturacion_Oxigeno), 1) AS spo2_promedio
    FROM logs_limpios
    WHERE Saturacion_Oxigeno < 90
    GROUP BY ID_Paciente
    ORDER BY alertas_criticas DESC, ID_Paciente
    LIMIT 5
""")
top5.show()

## 5. Query Big Data 3 — Clasificación automática de riesgo (UDF lambda)

Una función **UDF lambda** evalúa *simultáneamente* la frecuencia cardíaca y la saturación de
oxígeno, y Spark la ejecuta de forma distribuida para inyectar la columna `Nivel_Riesgo`:

| Nivel | Criterio |
|---|---|
| Crítico | SpO2 < 90 % **o** FC > 130 **o** FC < 40 |
| Moderado | SpO2 90–93 % **o** FC 101–130 **o** FC 40–49 |
| Bajo | todo lo demás |

In [ ]:
clasificar_riesgo = F.udf(
    lambda fc, spo2: None if fc is None or spo2 is None else (
        "Crítico" if spo2 < 90 or fc > 130 or fc < 40 else
        "Moderado" if spo2 < 94 or fc > 100 or fc < 50 else
        "Bajo"),
    T.StringType())

logs_refinados = logs_limpios.withColumn(
    "Nivel_Riesgo", clasificar_riesgo(F.col("Frecuencia_Cardiaca"), F.col("Saturacion_Oxigeno")))

logs_refinados.groupBy("Nivel_Riesgo").count().orderBy(F.desc("count")).show()
logs_refinados.filter(F.col("Nivel_Riesgo") == "Crítico").show(5, truncate=False)

## 6. Nivel de riesgo sostenido por paciente

La UDF clasifica **cada lectura**. Para el nivel del paciente no alcanza con una lectura aislada
(un pico puede ser ruido del sensor), así que se mira la proporción durante el período:

| Nivel del paciente | Regla |
|---|---|
| Crítico | al menos el 25 % de sus lecturas fueron críticas |
| Moderado | al menos el 25 % fueron moderadas o críticas |
| Bajo | cualquier otro caso |

En el dashboard este resultado se cruza con la internación de cada paciente y se guarda en
`asignacionCama.nivelRiesgo` (lo único que vuelve a SQL Server; los logs no se guardan).

In [ ]:
UMBRAL = 0.25
riesgo_paciente = (logs_refinados.groupBy("ID_Paciente")
    .agg(F.count(F.lit(1)).alias("lecturas"),
         F.sum(F.when(F.col("Nivel_Riesgo") == "Crítico", 1).otherwise(0)).alias("criticas"),
         F.sum(F.when(F.col("Nivel_Riesgo") == "Moderado", 1).otherwise(0)).alias("moderadas"))
    .withColumn("nivelRiesgo",
                F.when(F.col("criticas") / F.col("lecturas") >= UMBRAL, "Crítico")
                 .when((F.col("criticas") + F.col("moderadas")) / F.col("lecturas") >= UMBRAL, "Moderado")
                 .otherwise("Bajo"))
    .orderBy(F.desc("criticas")))

riesgo_paciente.show(truncate=False)
riesgo_paciente.toPandas().to_csv("riesgo_por_paciente.csv", index=False)

## 7. Exportación corporativa a Parquet

El DataFrame refinado (limpio + `Nivel_Riesgo`) se guarda en formato **Parquet** con
`.write.parquet()` y se vuelve a leer para comprobar que el archivo es válido y conserva el esquema.

In [ ]:
RUTA_PARQUET = "logs_refinados.parquet"
logs_refinados.write.mode("overwrite").parquet(RUTA_PARQUET)

verificacion = spark.read.parquet(RUTA_PARQUET)
verificacion.printSchema()
print("Registros en el Parquet:", verificacion.count())
!ls -la logs_refinados.parquet

## Extra: visualización del resultado (pandas + matplotlib)

In [ ]:
import matplotlib.pyplot as plt

riesgo = logs_refinados.groupBy("Nivel_Riesgo").count().toPandas().set_index("Nivel_Riesgo")["count"]
top = top5.toPandas()

fig, (a1, a2) = plt.subplots(1, 2, figsize=(12, 4))
a1.pie(riesgo, labels=riesgo.index, autopct="%1.0f%%", wedgeprops={"width": 0.4})
a1.set_title("Nivel de riesgo (Query BD 3)")
a2.barh(top["ID_Paciente"][::-1], top["alertas_criticas"][::-1], color="#0f6e66")
a2.set_title("Top 5 alertas críticas (Query BD 2)")
a2.set_xlabel("lecturas con SpO2 < 90 %")
plt.tight_layout()
plt.show()

### Descargar los resultados (opcional, solo en Colab)
Comprime el Parquet y el CSV para entregarlos junto al proyecto.

In [ ]:
import shutil
shutil.make_archive("resultados_bigdata", "zip", ".", "logs_refinados.parquet")
try:
    from google.colab import files
    files.download("resultados_bigdata.zip")
    files.download("logs_monitoreo_pacientes.csv")
    files.download("riesgo_por_paciente.csv")
except ImportError:
    print("Fuera de Colab: los archivos quedaron en la carpeta actual.")